# 03. 심화: calibration, ROC AUC와 길이 효과

목표: 여러 문서를 Monte Carlo로 생성해 null score 분포, fixed threshold의 false-positive rate, marked score 분포와 ROC AUC를 계산한다.

In [ ]:
import hashlib, hmac, math, random, statistics

VOCAB = [f't{i}' for i in range(16)]
GAMMA, CONTEXT = 0.5, 2

def greens(key: bytes, prefix: tuple[str, ...]) -> set[str]:
    raw = ' '.join(prefix).encode()
    ranked = sorted((hmac.new(key, raw + b'|' + t.encode(), hashlib.sha256).digest(), t) for t in VOCAB)
    return {t for _, t in ranked[:8]}

def document(key: bytes, n: int, delta: float, seed: int) -> list[str]:
    rng, result = random.Random(seed), []
    for _ in range(n):
        preferred = greens(key, tuple(result[-CONTEXT:]))
        result.append(rng.choices(VOCAB, [math.exp(delta if t in preferred else 0) for t in VOCAB])[0])
    return result

def score(tokens: list[str], key: bytes) -> float:
    count = sum(t in greens(key, tuple(tokens[max(0, i-CONTEXT):i])) for i, t in enumerate(tokens))
    n = len(tokens)
    return (count - GAMMA*n) / math.sqrt(n*GAMMA*(1-GAMMA))

def auc(negative: list[float], positive: list[float]) -> float:
    # positive score가 negative보다 클 확률. tie는 0.5로 센다.
    wins = sum((p > n) + 0.5*(p == n) for p in positive for n in negative)
    return wins / (len(negative) * len(positive))

In [ ]:
key = b'calibration-key'
threshold = 4.0
for length in [100, 400, 1500]:
    null = [score(document(key, length, delta=0.0, seed=i), key) for i in range(120)]
    marked = [score(document(key, length, delta=0.65, seed=10_000+i), key) for i in range(120)]
    fpr = sum(value >= threshold for value in null) / len(null)
    tpr = sum(value >= threshold for value in marked) / len(marked)
    print(
        f'n={length:4}  null μ={statistics.mean(null):5.2f}  marked μ={statistics.mean(marked):5.2f}  '
        f'FPR@z4={fpr:5.1%}  TPR@z4={tpr:5.1%}  AUC={auc(null, marked):.3f}'
    )

길이가 늘면 null z-score의 규모는 비슷하게 표준화되지만 marked 평균 z-score는 대략 `√N`으로 커진다. 따라서 같은 약한 bias도 긴 문서에서 잘 검출된다.

In [ ]:
# 여러 span을 훑을 때 생기는 multiple-testing 효과를 toy로 확인한다.
docs = [document(key, 1200, delta=0.0, seed=30_000+i) for i in range(100)]
for windows_per_doc in [1, 3, 10]:
    false_flags = 0
    for tokens in docs:
        width = len(tokens) // windows_per_doc
        span_scores = [score(tokens[i*width:(i+1)*width], key) for i in range(windows_per_doc)]
        false_flags += max(span_scores) >= 3.0
    print(f'spans={windows_per_doc:2}, document-level false flag rate={false_flags/len(docs):.1%}')

## 실무 체크리스트

- threshold는 문서 길이, 언어, domain, tokenizer와 detector version별로 calibration한다.
- 여러 span/key/model을 탐색하면 document-level false positive가 커지므로 보정한다.
- ROC AUC가 높아도 운영 threshold의 FPR/TPR과 base rate가 나쁠 수 있다.
- score만 저장하지 말고 적용 범위, 미지원 조건, text hash와 calibration metadata를 남긴다.
- 이 notebook의 synthetic token은 language quality와 semantic preservation을 평가하지 않는다.